   # 01 Model
   Builds the modelling dataset from the SQL files in sql/, then trains and evaluates the bad review model.

In [1]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect("../data/olist.duckdb")


## Setting up the SQL Files

In [ ]:
# run a sql file against the database
def run_sql(file):
    con.execute(open(f"../sql/{file}").read())

## Clean orders
One row per delivered order with its items, payments and earliest review.

In [ ]:
run_sql("clean_orders.sql")
# checking if each table has one row per order
for t in ["first_review", "order_items_agg", "order_payments_agg", "clean_orders"]:
    rows, orders = con.execute(f"select count(*), count(distinct order_id) from {t}").fetchone()
    print(f"{t:} rows={rows:}  orders={orders:}")

first_review rows=98673  orders=98673
order_items_agg rows=98666  orders=98666
order_payments_agg rows=99440  orders=99440
clean_orders rows=95658  orders=95658


Every table has one row per order, so joins won't duplicate orders. clean_orders keeps 95,658 delivered orders with valid dates and a review.

## Seller history
Each seller's earlier late deliveries and bad reviews, counting only outcomes already known at the order's handover.

In [17]:
run_sql("seller_history.sql")

# orders with seller history, and orders whose seller has no earlier known orders
print(con.execute("select count(*) from seller_history").fetchone()[0], "orders with history")
print(con.execute("""
    select count(*) from clean_orders o
    left join seller_history h on o.order_id = h.order_id
    where h.order_id is null
""").fetchone()[0], "orders with no history")

92620 orders with history
3038 orders with no history


In [ ]:
# raw orders for one seller: handover, delivery and review dates with outcomes with an example seller
con.execute("""
    select order_id, handover_ts, delivered_ts, promised_ts,
           (delivered_ts > promised_ts)::int as late,
           review_answer_timestamp, review_score,
           (review_score <= 2)::int as bad_review
    from clean_orders
    where seller_id = 'c826c40d7b19f62a09e2d7c5e7295ee2'
    order by handover_ts
    limit 6
""").df()

,order_id,handover_ts,delivered_ts,promised_ts,late,review_answer_timestamp,review_score,bad_review
0,9d7f2c9f33008498700ffdc3162878fb,2017-08-28 19:03:48,2017-08-31 21:00:31,2017-09-18,0,2017-09-04 20:46:52,5,0
1,392240bf6b91914830b77ae8572cc0b7,2017-09-05 21:53:43,2017-09-19 19:53:04,2017-09-28,0,2017-09-21 02:07:57,1,1
2,c501bcc74df5ad09bcabd7774b6ae383,2017-09-19 20:24:36,2017-10-05 11:33:35,2017-10-06,0,2017-10-09 11:54:37,5,0
3,c1e50c50037af907b9099da2c948e109,2017-09-25 20:32:44,2017-10-02 18:07:56,2017-10-18,0,2017-10-04 16:18:34,4,0
4,51427a59ed7cf06820364d94af7a6f7a,2017-09-26 20:33:02,2017-10-11 15:39:15,2017-10-13,0,2017-10-16 17:32:20,4,0
5,8c4bbf2fb218d1ac2fc4fc082c238d64,2017-10-06 21:03:55,2017-10-23 21:11:32,2017-11-07,0,2017-10-24 23:32:51,4,0


In [6]:
# history for one seller over time: counts grow only once outcomes are known
con.execute("""
    select o.handover_ts, h.seller_past_orders, h.seller_past_late, h.seller_past_reviews, h.seller_past_bad
    from clean_orders o
    left join seller_history h on o.order_id = h.order_id
    where o.seller_id = 'c826c40d7b19f62a09e2d7c5e7295ee2'
    order by o.handover_ts
    limit 6
""").df()

,handover_ts,seller_past_orders,seller_past_late,seller_past_reviews,seller_past_bad
0,2017-08-28 19:03:48,<NA>,<NA>,<NA>,<NA>
1,2017-09-05 21:53:43,1,0,1,0
2,2017-09-19 20:24:36,2,0,1,0
3,2017-09-25 20:32:44,2,0,2,1
4,2017-09-26 20:33:02,2,0,2,1
5,2017-10-06 21:03:55,4,0,3,1


The history only counts what was already known at each handover. When this seller's third order was handed over on 19 Sep, the second order had just been delivered but its 1-star review was only written on 21 Sep, so it is not counted yet; it appears from the fourth order onwards. The model therefore only uses information that would really be available at the time of prediction.

## Train and test periods
The model is trained on earlier orders and tested on later ones, so the evaluation reflects how it would be used on future orders.

| Period | Handover dates | Used for |
|---|---|---|
| Train | 1 Jan 2017 to 31 Mar 2018 | fitting the model and calculating the smoothing averages |
| Test | 1 Apr 2018 to 31 Aug 2018 | evaluating the model only |

The cutoff of 1 April 2018 gives roughly two thirds of orders for training and one third for testing. Anything calculated from the data before modelling, such as the smoothing averages below, uses training orders only, so nothing from the test period influences the model.

In [14]:
# average late and bad review rates in the training period
con.execute("""
    select count(*) as orders,
           sum((delivered_ts > promised_ts)::int) as late_orders,
           round(100 * avg((delivered_ts > promised_ts)::int), 1) as late_pct,
           sum((review_score <= 2)::int) as bad_reviews,
           round(100 * avg((review_score <= 2)::int), 1) as bad_review_pct
    from clean_orders
    where handover_ts >= timestamp '2017-01-01'
      and handover_ts < timestamp '2018-04-01'
""").df()

,orders,late_orders,late_pct,bad_reviews,bad_review_pct
0,62833,5607.0,8.9,8734.0,13.9


8.9% of training orders were delivered late and 13.9% got a bad review. These are rounded to 9% and 14% and used as the smoothing averages in `feature_table.sql`

## Smoothing formula

For an order handed over at time $t$, by seller $s$:

$$
\hat{r}_{s,t} = \frac{x_{s,t} + k \cdot \bar{r}}{n_{s,t} + k}
$$

| Symbol | Meaning | Value |
|---|---|---|
| $n_{s,t}$ | seller's earlier orders whose outcome was known at $t$ | from seller_history |
| $x_{s,t}$ | how many of those were late (or got a bad review) | from seller_history |
| $\bar{r}$ | average rate in the training period | 0.09 late, 0.14 bad review |
| $k$ | weight given to the average, in "orders" | 10 |

It can also be written as a weighted average of the seller's own rate and the overall average:

$$
\hat{r}_{s,t} = w \cdot \frac{x_{s,t}}{n_{s,t}} + (1 - w) \cdot \bar{r}, \qquad w = \frac{n_{s,t}}{n_{s,t} + k}
$$

The more orders a seller has, the closer $w$ gets to 1 and the more their own rate counts. With no orders, $w = 0$ and the rate equals the average. Statistically, this is the posterior mean of a Beta-Binomial model with a $\text{Beta}(k\bar{r},\ k(1-\bar{r}))$ prior, the same idea as a Bayesian average or smoothed target encoding.

**Worked example (late rate, $\bar{r} = 0.09$, $k = 10$)**

A seller has 20 earlier orders known at handover, of which 6 arrived late.

1. Raw rate: $6 / 20 = 30\%$
2. Weight on the seller's own rate: $w = 20 / (20 + 10) = 0.67$
3. Smoothed rate: $0.67 \times 30\% + 0.33 \times 9\% = 23.0\%$

Same result from the first formula: $(6 + 10 \times 0.09) / (20 + 10) = 6.9 / 30 = 23.0\%$.

| Seller | $n$ | $x$ | Raw rate | $w$ | Smoothed |
|---|---|---|---|---|---|
| New | 0 | 0 | none | 0.00 | 9.0% |
| 1 order, late | 1 | 1 | 100% | 0.09 | 17.3% |
| 2 orders, on time | 2 | 0 | 0% | 0.17 | 7.5% |
| 20 orders, 6 late | 20 | 6 | 30% | 0.67 | 23.0% |
| 500 orders, 40 late | 500 | 40 | 8% | 0.98 | 8.0% |

A seller with one late order out of one is treated as somewhat worse than average (17%), not as always late (100%). A seller with 500 orders keeps their real rate.

## Feature table
One row per delivered order, observed at the moment the seller hands the parcel to the carrier. Every feature is known at that moment.

| Group | Features | Why |
|---|---|---|
| Seller dispatch | dispatch_days_late (capped at ±30 days), dispatch_late, days_to_handover | late dispatch doubles the bad review rate |
| Delivery promise | promised_days, days_left_to_promise | few days left at handover means a late delivery is likely |
| Basket | n_items, n_sellers, total_price, freight_ratio | larger or costly-to-ship orders have more that can go wrong |
| Geography | same_state, customer_state | orders crossing states travel further |
| Product | category ('unknown' if missing, Portuguese name if no translation), missing_listing, photos | some categories and incomplete listings get more bad reviews |
| Seller history | seller_past_orders, seller_past_late_rate, seller_past_bad_rate (smoothed) | a seller's track record predicts future problems |
| Target | bad_review (1 = 1-2 stars) | what the model predicts |

Not included, because they are only known after handover:
- the actual delivery date, whether the order arrived late, and the review text. 
- Payment data is left out as it adds nothing beyond price and freight. 
- Only handovers between Jan 2017 and Aug 2018 are kept.

In [21]:
run_sql("feature_table.sql")
features = con.execute("select * from feature_table").df()

# size, bad review rate, date range and missing values
print(features.shape )
print('Date range:', features.handover_ts.min(), "to", features.handover_ts.max())
print("missing values:", features.isnull().sum().sum())
features.head()

(95392, 20)
Date range: 2017-01-06 12:43:41 to 2018-08-31 15:25:00
missing values: 0


,order_id,handover_ts,dispatch_days_late,dispatch_late,days_to_handover,promised_days,days_left_to_promise,n_items,n_sellers,total_price,freight_ratio,same_state,customer_state,category,missing_listing,photos,seller_past_orders,seller_past_late_rate,seller_past_bad_rate,bad_review
0,36af5b0b0ededbf6f30d1d809f05536b,2018-06-04 14:10:00,-7.250000,0,0.750000,32,30.416667,1,1,115.0,0.136174,1,MG,fashion_bags_accessories,0,6,30,0.072500,0.152381,0
1,520563861deb0acf3b7ab8295ced6e01,2018-01-18 12:52:07,-3.625000,0,3.833333,32,27.500000,1,1,139.0,0.215899,0,BA,auto,0,1,272,0.070567,0.072857,0
2,459aa8ae0e09ae140ba9f75c1f429524,2017-04-11 14:28:46,-5.583333,0,2.791667,30,26.416667,1,1,44.0,0.364773,0,RJ,baby,0,2,6,0.056250,0.087500,0
3,474cefe70ac457aaa468b62e07818c5e,2018-01-27 01:42:53,-4.375000,0,38.666667,56,16.958333,1,1,339.0,0.031180,1,SP,consoles_games,0,2,19,0.203448,0.174194,0
4,df85286f77ee8760333080a6b3bb97d8,2018-02-14 19:51:40,-0.875000,0,3.166667,26,22.208333,1,1,119.9,0.121518,0,SP,health_beauty,0,2,77,0.033333,0.051765,0


In [24]:
# sellers with no earlier orders should get exactly the smoothing averages
features[features.seller_past_orders == 0][["seller_past_late_rate", "seller_past_bad_rate"]].value_counts()

seller_past_late_rate  seller_past_bad_rate
0.09                   0.140000                4340
                       0.218182                   9
                       0.127273                   2
                       0.283333                   2
                       0.385714                   1
Name: count, dtype: int64